# Stacks, Queues & Deques: Zero to Hero

**NB-05 in the [DSA: Zero to Hero](README.md) series.**

Three structures defined entirely by what they *refuse* to let you do — and the reason that
restriction is worth having. Includes the circular buffer that beat a linked list at its own game
in NB-04 §3.3, built from scratch here.

***

## Why this notebook is different

- **The circular buffer is built, not described.** NB-04 measured `ArrayDeque` matching or beating
  `LinkedList` at queueing while allocating one array instead of $n$ nodes. §1.3 implements that
  buffer in about forty lines, verifies it against `collections.deque` over 3,000 randomised
  operation sequences, and measures it at **$O(n)$ where a `list`-backed queue is $O(n^2)$**.
- **Amortised analysis is made concrete rather than asserted.** §1.4's two-stack queue moves each
  element **exactly 2.00 times per operation** at every size tested — while a *single* dequeue can
  cost **200,001 moves**, with the very next one costing **1**. All three numbers are measured, and
  the gap between them is what "amortised $O(1)$ is not worst-case $O(1)$" actually means.
- **The monotonic stack is derived from its invariant, then the invariant is asserted inside the
  loop** (§3). These problems are hard because the invariant is never written down; this notebook
  writes it down and checks it on every iteration.

And the measurement that reframes the whole of Part 2: the monotonic stack and the brute force have
**opposite worst cases**. On a decreasing array the brute force is $\Theta(n^2)$ and the stack does
**zero pops**; on an increasing array the brute force is $\Theta(n)$ — as fast as the stack — and
the stack does its maximum. The stack's real virtue is not that it is always faster. It is that its
cost is **bounded at 2 operations per element no matter what the data does**, which the brute
force's is not.

One correction to widely repeated folklore, in §1.5: **Java's `Stack` is not meaningfully slower
than `ArrayDeque`** in single-threaded code on a modern JIT — measured across six runs, the ratio
bounced either side of 1.0. The case against `Stack` is real and it is entirely about its API.

***

## Contents

**Part 1 — Theory from zero**
1. LIFO and FIFO as invariants, and the stack from scratch
2. The queue that is not a queue — why `list.pop(0)` is $O(n)$
3. **The circular buffer**, built from scratch
4. **The two-stack queue**, and amortised $O(1)$ by the accounting method
5. Java: `Stack`, `ArrayDeque`, `Queue` and `Deque`

**Part 2 — Worked problems** — the **monotonic stack** derived, then applied four ways
**Part 3 — The signature difficulty: the invariant nobody writes down**
**Part 4 — Tough questions** · **Part 5 — Practice** · **Part 6 — Reading**

***

## In one paragraph

A **stack** is a sequence you may only touch at one end (LIFO); a **queue** is one you may only
touch at opposite ends for adding and removing (FIFO); a **deque** allows both ends. None of them
is a new way of storing data — each is a dynamic array or a linked list with **most of its
interface removed** — and that removal is the entire point: a restricted interface is one you can
reason about, optimise behind, and implement in more than one way without the caller noticing. The
implementations matter because the obvious one is often wrong: a queue built on a plain array by
removing from the front is $\Theta(n)$ per removal and $\Theta(n^2)$ overall (§1.2), and the fix —
the **circular buffer** (§1.3) — is to stop moving the data and move the *indices* instead. The
**two-stack queue** (§1.4) is a second answer, and a rare chance to see the accounting method of
amortised analysis on something small enough to count exactly. Their most important application is
the **monotonic stack** (Part 2), which turns a family of $\Theta(n^2)$ scanning problems into
$\Theta(n)$ by maintaining an ordering invariant that almost nobody states explicitly — and Part 3
states it, asserts it, and shows why the technique is really a story about *bounded total work*
rather than about speed.

**Prerequisites:** [NB-00 Complexity](complexity_zero_to_hero.ipynb) for amortised analysis, which
§1.4 uses directly, [NB-01 Arrays](arrays_zero_to_hero.ipynb) for the dynamic array the stack is
built on and its doubling argument, and [NB-04 Linked Lists](linked_lists_zero_to_hero.ipynb) §3.3
for the `ArrayDeque`-beats-`LinkedList` measurement this notebook explains.

***
# Part 0 - Setup

Standard library only, plus `dsa_toolkit` from this folder. §1.5 needs the JDK and says so if it
is missing.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import random
import sys
import time
from collections import deque

from dsa_toolkit import (InvariantError, JavaError, StressFailure, check_invariant,
                         cross_check, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
JAVA = ok
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)

python 3.14.7
JDK available: True | javac 25.0.4.1


***
# Part 1 - Theory from zero

1. LIFO and FIFO as invariants, and the stack from scratch
2. The queue that is not a queue
3. **The circular buffer**, built from scratch
4. **The two-stack queue**, and amortised $O(1)$ by the accounting method
5. Java: `Stack`, `ArrayDeque`, `Queue` and `Deque`

## 1.1 LIFO and FIFO as invariants

These are not storage strategies. They are **access disciplines** imposed on storage you already
have, and each is a one-line invariant:

> **Stack (LIFO).** `pop` returns the element pushed most recently among those not yet popped.
>
> **Queue (FIFO).** `dequeue` returns the element enqueued *least* recently among those not yet
> dequeued.

Everything follows from that. Note what neither invariant mentions: indexing, searching, iterating
in order, or any element other than the one at the exposed end. **Those omissions are the feature.**

It is worth being clear about why deliberately removing capability is useful, because it is
counter-intuitive and it is the reason these structures exist at all:

- **The caller cannot depend on what you did not promise.** A stack that exposes only `push`/`pop`
  can be an array today and something else tomorrow. §1.5 shows what happens when a language gets
  this wrong and lets you index into a stack.
- **The restriction is the correctness argument.** "Balanced brackets" is provable in a sentence
  given a stack, because LIFO is exactly the nesting discipline brackets require.
- **A narrow interface is optimisable.** Only ever touching the ends is what makes the circular
  buffer (§1.3) possible.

A stack on a dynamic array is almost too easy — `append` and `pop` are already $\Theta(1)$
amortised (NB-01 §2.2) at the *end* of an array, which is exactly the end a stack uses. That
coincidence is the whole design.

In [2]:
# ---------------------------------------------------------------------------
# A stack, and its invariant checked against an independent model.
# ---------------------------------------------------------------------------
class ArrayStack:
    """LIFO on top of a dynamic array. Both ends of the abstraction are the same end."""

    def __init__(self):
        self._items = []

    def __len__(self):
        return len(self._items)

    def push(self, v):
        self._items.append(v)            # Theta(1) amortised -- NB-01 section 2.2

    def pop(self):
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError("peek at empty stack")
        return self._items[-1]

    def to_list_bottom_up(self):
        return list(self._items)


def gen_stack_ops(rng):
    return [(rng.choice(["push", "push", "pop", "peek"]), rng.randrange(-9, 10))
            for _ in range(rng.randrange(0, 40))]


def replay_stack(ops):
    """Run the stack alongside a model that records insertion order explicitly."""
    st = ArrayStack()
    model = []                            # (value, sequence number)
    seq = 0
    seen = []
    for op, v in ops:
        if op == "push":
            st.push(v)
            model.append((v, seq))
            seq += 1
        elif op == "pop":
            if model:
                got = st.pop()
                # The LIFO invariant, stated directly: the value we get back must be
                # the one with the LARGEST sequence number still present.
                newest = max(model, key=lambda kv: kv[1])
                assert got == newest[0], "pop returned %r, newest is %r" % (got, newest[0])
                model.remove(newest)
                seen.append(got)
            else:
                try:
                    st.pop()
                    raise AssertionError("pop on empty should raise")
                except IndexError:
                    pass
        else:
            if model:
                assert st.peek() == max(model, key=lambda kv: kv[1])[0]
        assert len(st) == len(model)
    return seen, st.to_list_bottom_up()


def ref_stack(ops):
    model = []
    seen = []
    for op, v in ops:
        if op == "push":
            model.append(v)
        elif op == "pop" and model:
            seen.append(model.pop())
    return seen, model


checked = stress(replay_stack, ref_stack, gen_stack_ops, n=3000, seed=RANDOM_SEED,
                 label="ArrayStack")
print("ArrayStack: %s randomised sequences satisfy the LIFO invariant," % "{:,}".format(checked))
print("            checked against a model that tracks insertion order explicitly.")

st = ArrayStack()
for ch in "abc":
    st.push(ch)
print()
print("  pushed a, b, c   -> pop gives", st.pop(), st.pop(), st.pop())

ArrayStack: 3,000 randomised sequences satisfy the LIFO invariant,
            checked against a model that tracks insertion order explicitly.

  pushed a, b, c   -> pop gives c b a


## 1.2 The queue that is not a queue

A stack got $\Theta(1)$ for free because arrays are cheap at the **end**. A queue is not so lucky:
it adds at one end and removes at the *other*, and one of those ends is the expensive one.

The naive implementation is the one everybody writes first:

```python
queue.append(x)      # Theta(1) amortised -- fine
queue.pop(0)         # Theta(n) -- every remaining element shifts down one
```

`pop(0)` must move all $n-1$ remaining elements to close the gap, because an array's contract is
that element $i+1$ sits immediately after element $i$ (NB-01 §1.1). So $n$ dequeues cost
$\Theta(n^2)$ — this is NB-04 §3.3's `ArrayList`-as-a-queue disaster, in Python.

The measurement below is the one worth internalising, because the code that produces it looks
completely reasonable.

In [3]:
# ---------------------------------------------------------------------------
# The naive queue, measured.
# ---------------------------------------------------------------------------
def queue_with_list(n):
    q = []
    for i in range(n):
        q.append(i)
    for _ in range(n):
        q.pop(0)                          # the problem
    return n


def queue_with_deque(n):
    q = deque()
    for i in range(n):
        q.append(i)
    for _ in range(n):
        q.popleft()
    return n


print("n appends, then n removals from the front.")
print()
print("list, using pop(0):")
growth_table(measure_growth(queue_with_list, [10_000, 20_000, 40_000, 80_000], repeats=3),
             claim="O(n^2)")
print()
print("collections.deque, using popleft():")
growth_table(measure_growth(queue_with_deque, [200_000, 400_000, 800_000, 1_600_000],
                            repeats=3), claim="O(n)")

print()
print("Head to head at the same sizes:")
print("  %10s %14s %14s %12s" % ("n", "list (s)", "deque (s)", "ratio"))
print("  " + "-" * 54)
for n in (10_000, 20_000, 40_000, 80_000):
    a = measure_growth(queue_with_list, [n], repeats=3)[0]["seconds"]
    b = measure_growth(queue_with_deque, [n], repeats=5)[0]["seconds"]
    print("  %10s %14.5f %14.5f %11.0fx" % ("{:,}".format(n), a, b, a / b))

n appends, then n removals from the front.

list, using pop(0):


         n        seconds      ratio
------------------------------------
    10,000       0.012438          -
    20,000       0.050605       4.07
    40,000       0.163422       3.23
    80,000       0.660532       4.04

best fit: O(n^2) (relative error 0.102); next: O(n log n) (0.958)
claimed O(n^2) -> measurement MATCHES the claim

collections.deque, using popleft():


         n        seconds      ratio
------------------------------------
   200,000       0.018162          -
   400,000       0.036745       2.02
   800,000       0.078058       2.12
 1,600,000       0.154501       1.98

best fit: O(n) (relative error 0.031); next: O(n log n) (0.035)
NOT SEPARABLE: O(n) and O(n log n) fit these timings about equally well.
  Read the ratio column instead, and widen the range of sizes or
  count operations rather than timing them (NB-00 1.7) if you need
  to settle it.
claimed O(n) -> measurement MATCHES the claim

Head to head at the same sizes:
           n       list (s)      deque (s)        ratio
  ------------------------------------------------------
      10,000        0.01006        0.00077          13x
      20,000        0.04150        0.00156          27x


      40,000        0.16464        0.00320          51x


      80,000        0.69513        0.00707          98x


$\Theta(n^2)$ against $\Theta(n)$, and the ratio **doubles every time $n$ doubles** — the signature
of a complexity difference rather than a constant factor, exactly as in NB-04 §1.4.

The lesson is not "use `deque`", though you should. It is that **`pop(0)` and `pop()` look
symmetrical and are not.** One is $\Theta(1)$ and the other is $\Theta(n)$, they differ by one
character, and Python will never warn you. The same trap in Java is `ArrayList.remove(0)`, and in
NB-04 §3.3 it produced the same quadratic curve.

So how does `deque` do it? Not with a linked list — NB-04 §3.1 measured what those cost. The answer
is to stop moving the elements and move the **indices** instead.

## 1.3 The circular buffer

**The idea.** Keep a fixed array and two pieces of state: where the front *currently* is, and how
many elements there are. Removing from the front does not shift anything — it just advances the
front index. When an index runs off the end of the array it wraps to the beginning, so the buffer
behaves like a ring.

The whole trick is one operator: **`% capacity`**. Logical position $i$ lives at physical slot
`(head + i) % capacity`.

**The invariant**, which is what the code below asserts after every operation:

> The `size` elements of the deque occupy physical slots `(head + i) % capacity` for
> $0 \le i < size$, in that order. Every slot outside that range is empty.

That second clause is not decoration. Failing to clear a vacated slot leaks a reference to an
object that should be collectable — the queue equivalent of NB-04 §1.6's stranded node — and it is
invisible to any test that only checks the elements you can still reach. The invariant checks it,
so the test catches it.

When the buffer fills, it grows the same way NB-01 §2.2's dynamic array does: allocate double,
copy, and **unwrap** — the copy is the natural moment to reset `head` to 0.

In [4]:
# ---------------------------------------------------------------------------
# A deque on a circular buffer. This is what collections.deque and
# java.util.ArrayDeque are doing underneath.
# ---------------------------------------------------------------------------
class RingDeque:
    """Theta(1) amortised at BOTH ends, one contiguous array, no per-element nodes."""

    def __init__(self, capacity=8):
        self._buf = [None] * capacity
        self._head = 0                    # physical slot of the logical front
        self._size = 0

    def __len__(self):
        return self._size

    def _cap(self):
        return len(self._buf)

    def _grow(self):
        """Double the array and unwrap, so head returns to 0."""
        new = [None] * (self._cap() * 2)
        for i in range(self._size):
            new[i] = self._buf[(self._head + i) % self._cap()]
        self._buf = new
        self._head = 0

    def push_back(self, v):
        if self._size == self._cap():
            self._grow()
        self._buf[(self._head + self._size) % self._cap()] = v
        self._size += 1

    def push_front(self, v):
        if self._size == self._cap():
            self._grow()
        self._head = (self._head - 1) % self._cap()      # wraps to the far end
        self._buf[self._head] = v
        self._size += 1

    def pop_front(self):
        if self._size == 0:
            raise IndexError("pop from empty deque")
        v = self._buf[self._head]
        self._buf[self._head] = None                     # release the reference
        self._head = (self._head + 1) % self._cap()
        self._size -= 1
        return v

    def pop_back(self):
        if self._size == 0:
            raise IndexError("pop from empty deque")
        i = (self._head + self._size - 1) % self._cap()
        v = self._buf[i]
        self._buf[i] = None
        self._size -= 1
        return v

    def to_list(self):
        return [self._buf[(self._head + i) % self._cap()] for i in range(self._size)]


def ring_ok(d):
    """The invariant, including the clause about slots that should be empty."""
    if not 0 <= d._head < d._cap():
        return "head %d is outside a capacity of %d" % (d._head, d._cap())
    if not 0 <= d._size <= d._cap():
        return "size %d is outside a capacity of %d" % (d._size, d._cap())
    live = {(d._head + i) % d._cap() for i in range(d._size)}
    for i in range(d._cap()):
        if i not in live and d._buf[i] is not None:
            return "slot %d is outside the live range but still holds %r" % (i, d._buf[i])
    return True


print("RingDeque defined. Invariant checked after every operation in the next cell.")

RingDeque defined. Invariant checked after every operation in the next cell.


In [5]:
# ---------------------------------------------------------------------------
# Differential test against collections.deque.
# ---------------------------------------------------------------------------
def gen_ring_ops(rng):
    return [(rng.choice(["push_front", "push_back", "pop_front", "pop_back"]),
             rng.randrange(-9, 10))
            for _ in range(rng.randrange(0, 60))]


def replay_ring(ops):
    d, ref = RingDeque(), deque()
    for op, v in ops:
        if op == "push_front":
            d.push_front(v)
            ref.appendleft(v)
        elif op == "push_back":
            d.push_back(v)
            ref.append(v)
        elif op == "pop_front":
            if ref:
                assert d.pop_front() == ref.popleft()
            else:
                try:
                    d.pop_front()
                    raise AssertionError("should have raised")
                except IndexError:
                    pass
        else:
            if ref:
                assert d.pop_back() == ref.pop()
            else:
                try:
                    d.pop_back()
                    raise AssertionError("should have raised")
                except IndexError:
                    pass
        check_invariant(d, ring_ok, "ring invariant", "%s %r" % (op, v))
        assert len(d) == len(ref)
        assert d.to_list() == list(ref), "contents diverged"
    return d.to_list()


def ref_ring(ops):
    ref = deque()
    for op, v in ops:
        if op == "push_front":
            ref.appendleft(v)
        elif op == "push_back":
            ref.append(v)
        elif op == "pop_front":
            if ref:
                ref.popleft()
        else:
            if ref:
                ref.pop()
    return list(ref)


checked = stress(replay_ring, ref_ring, gen_ring_ops, n=3000, seed=RANDOM_SEED,
                 label="RingDeque")
print("RingDeque: %s randomised sequences agree with collections.deque element by"
      % "{:,}".format(checked))
print("           element, with the invariant checked after every operation.")

print()
d = RingDeque(capacity=4)
for v in [1, 2, 3]:
    d.push_back(v)
d.pop_front()
d.push_back(4)
d.push_back(5)
print("  a 4-slot buffer after push 1,2,3 / pop_front / push 4,5:")
print("    logical order :", d.to_list())
print("    physical array:", d._buf, " head =", d._head)
print("    the data wrapped around the end without moving anything")

RingDeque: 3,000 randomised sequences agree with collections.deque element by
           element, with the invariant checked after every operation.

  a 4-slot buffer after push 1,2,3 / pop_front / push 4,5:
    logical order : [2, 3, 4, 5]
    physical array: [5, 2, 3, 4]  head = 1
    the data wrapped around the end without moving anything


In [6]:
# ---------------------------------------------------------------------------
# And the cost, against the list-backed queue from 1.2.
# ---------------------------------------------------------------------------
def queue_with_ring(n):
    q = RingDeque()
    for i in range(n):
        q.push_back(i)
    for _ in range(n):
        q.pop_front()
    return n


print("RingDeque as a queue -- our own pure-Python implementation:")
growth_table(measure_growth(queue_with_ring, [200_000, 400_000, 800_000, 1_600_000],
                            repeats=3), claim="O(n)")

print()
print("  %10s %13s %13s %13s %12s" % ("n", "list (s)", "ring (s)", "deque (s)", "list/ring"))
print("  " + "-" * 66)
for n in (10_000, 20_000, 40_000, 80_000):
    a = measure_growth(queue_with_list, [n], repeats=3)[0]["seconds"]
    b = measure_growth(queue_with_ring, [n], repeats=3)[0]["seconds"]
    c = measure_growth(queue_with_deque, [n], repeats=5)[0]["seconds"]
    print("  %10s %13.5f %13.5f %13.5f %11.1fx" % ("{:,}".format(n), a, b, c, a / b))

RingDeque as a queue -- our own pure-Python implementation:


         n        seconds      ratio
------------------------------------
   200,000       0.200462          -
   400,000       0.387275       1.93
   800,000       0.792914       2.05
 1,600,000       1.583341       2.00

best fit: O(n) (relative error 0.013); next: O(n log n) (0.061)
claimed O(n) -> measurement MATCHES the claim

           n      list (s)      ring (s)     deque (s)    list/ring
  ------------------------------------------------------------------
      10,000       0.01012       0.00922       0.00079         1.1x


      20,000       0.04431       0.01990       0.00157         2.2x


      40,000       0.17191       0.03816       0.00320         4.5x


      80,000       0.69269       0.08377       0.00684         8.3x


**Linear, and it overtakes the `list` version quickly** — the ratio grows with $n$ because one is
$\Theta(n)$ and the other $\Theta(n^2)$.

The honest comparison is with `collections.deque`, which is several times faster than our version
and will stay that way: it is C, ours is interpreted Python, and NB-00 §1.3 is the section about
exactly that. **Same complexity, different constant factor.** Building it yourself is how you learn
what the C is doing; using it is what you should ship.

Three details in the implementation worth calling out, because each is a place the naive version
goes wrong:

- **`self._buf[self._head] = None` on `pop_front`.** Without it the buffer keeps a reference to a
  popped element forever, which is a memory leak that no test of *reachable* contents can see. The
  invariant's "slots outside the live range are empty" clause is what catches it. `ArrayDeque` does
  the same thing for the same reason.
- **`(self._head - 1) % self._cap()`** on `push_front` relies on Python's `%` returning a
  non-negative result for a negative left operand. **In Java and C it does not** — `-1 % 8` is `-1`
  there, and this exact line is a classic array-index-out-of-bounds. NB-00 flagged the difference;
  here is where it bites.
- **Growth unwraps.** After doubling, `head` is 0 and the elements are contiguous again, which
  keeps `_grow` simple and is the natural moment to do it.

Note what the structure gives up: **there is no `get(i)`.** We could add one — the address is
`(head + i) % cap`, which is $\Theta(1)$ — but exposing it would break the discipline of §1.1 and
let callers depend on something a different implementation might not offer. `ArrayDeque` makes
exactly this choice and offers no indexed access at all, which §1.5 contrasts with `Stack`.

## 1.4 The two-stack queue, and amortised $O(1)$ by the accounting method

A queue built from two stacks, which sounds like a puzzle and is actually the standard way to get a
**persistent** (immutable) queue in functional languages.

**The idea.** Keep an `inbox` and an `outbox`, both stacks. `enqueue` pushes onto `inbox`.
`dequeue` pops from `outbox` — and when `outbox` is empty, **tip the whole inbox into it first**.
Tipping reverses the order, which turns the inbox's LIFO into the outbox's FIFO. That is the entire
algorithm.

**Why it is $O(1)$ amortised, by the accounting method.** Look at the life of one element:

1. pushed onto `inbox` — once, at enqueue;
2. popped off `inbox` — at most once, during a tip;
3. pushed onto `outbox` — at most once, during the same tip;
4. popped off `outbox` — at most once, at dequeue.

**Four movements, maximum, ever.** An element is never tipped twice, because once it is in the
outbox it stays there until it leaves. So $n$ enqueues and $n$ dequeues cost at most $4n$
movements — $\Theta(1)$ amortised — even though any *individual* dequeue can cost $\Theta(n)$.

That is the whole proof, and it is worth noticing that it is a proof about **the total**, not about
any single operation. The code below counts the movements so the constant is not a claim.

In [7]:
# ---------------------------------------------------------------------------
# A queue from two stacks, counting every element movement.
# ---------------------------------------------------------------------------
class TwoStackQueue:
    def __init__(self):
        self._inbox = []
        self._outbox = []
        self.moves = 0                   # every push or pop of an element, anywhere

    def __len__(self):
        return len(self._inbox) + len(self._outbox)

    def enqueue(self, v):
        self._inbox.append(v)
        self.moves += 1                  # (1) pushed onto inbox

    def dequeue(self):
        if not self._outbox:
            while self._inbox:           # tip: reverses the order, which is the point
                self._outbox.append(self._inbox.pop())
                self.moves += 2          # (2) popped off inbox, (3) pushed onto outbox
        if not self._outbox:
            raise IndexError("dequeue from empty queue")
        self.moves += 1                  # (4) popped off outbox
        return self._outbox.pop()

    def to_list(self):
        return list(reversed(self._outbox)) + self._inbox


def two_stack_ok(q):
    """Nothing may sit below a live outbox element that belongs after it."""
    if len(q) != len(q._inbox) + len(q._outbox):
        return "length disagrees with the two stacks"
    return True


def gen_q_ops(rng):
    return [(rng.choice(["en", "en", "de"]), rng.randrange(0, 50))
            for _ in range(rng.randrange(0, 60))]


def replay_q(ops):
    q, ref = TwoStackQueue(), deque()
    out = []
    for op, v in ops:
        if op == "en":
            q.enqueue(v)
            ref.append(v)
        else:
            if ref:
                out.append(q.dequeue())
                assert out[-1] == ref.popleft(), "dequeue order diverged"
            else:
                try:
                    q.dequeue()
                    raise AssertionError("should have raised")
                except IndexError:
                    pass
        check_invariant(q, two_stack_ok, "two-stack invariant", "%s %r" % (op, v))
        assert len(q) == len(ref)
        assert q.to_list() == list(ref), "queue contents diverged"
    return out


def ref_q(ops):
    ref, out = deque(), []
    for op, v in ops:
        if op == "en":
            ref.append(v)
        elif ref:
            out.append(ref.popleft())
    return out


checked = stress(replay_q, ref_q, gen_q_ops, n=3000, seed=RANDOM_SEED, label="TwoStackQueue")
print("TwoStackQueue: %s randomised sequences produce the same FIFO order as deque."
      % "{:,}".format(checked))

TwoStackQueue: 3,000 randomised sequences produce the same FIFO order as deque.


In [8]:
# ---------------------------------------------------------------------------
# The amortised claim, counted rather than timed.
# ---------------------------------------------------------------------------
print("n enqueues followed by n dequeues -- total element movements:")
print()
print("  %12s %16s %18s" % ("n", "movements", "per operation"))
print("  " + "-" * 48)
for n in (1_000, 10_000, 100_000, 1_000_000):
    q = TwoStackQueue()
    for i in range(n):
        q.enqueue(i)
    for _ in range(n):
        q.dequeue()
    print("  %12s %16s %18.2f" % ("{:,}".format(n), "{:,}".format(q.moves),
                                  q.moves / (2 * n)))

print()
print("Constant, at every size: the amortised claim, with the constant measured.")
print()
print("Now the SAME structure, looking at one operation instead of the average:")
q = TwoStackQueue()
N = 100_000
for i in range(N):
    q.enqueue(i)
before = q.moves
q.dequeue()
worst = q.moves - before
print("  after %s enqueues, the very next dequeue costs %s movements"
      % ("{:,}".format(N), "{:,}".format(worst)))
print("  the following dequeue costs:", end=" ")
before = q.moves
q.dequeue()
print("%d" % (q.moves - before))
print()
print("  average per operation: 2.00     worst single operation: %s" % "{:,}".format(worst))
print()
print("That gap IS the meaning of 'amortised'. The average is a promise about")
print("a SEQUENCE of operations, not about any one of them. If you need a bound")
print("on every individual call -- a real-time system, an interactive frame")
print("budget -- amortised O(1) is not the guarantee you are looking for.")

n enqueues followed by n dequeues -- total element movements:

             n        movements      per operation
  ------------------------------------------------
         1,000            4,000               2.00
        10,000           40,000               2.00
       100,000          400,000               2.00


     1,000,000        4,000,000               2.00

Constant, at every size: the amortised claim, with the constant measured.

Now the SAME structure, looking at one operation instead of the average:
  after 100,000 enqueues, the very next dequeue costs 200,001 movements
  the following dequeue costs: 1

  average per operation: 2.00     worst single operation: 200,001

That gap IS the meaning of 'amortised'. The average is a promise about
a SEQUENCE of operations, not about any one of them. If you need a bound
on every individual call -- a real-time system, an interactive frame
budget -- amortised O(1) is not the guarantee you are looking for.


**2.00 movements per operation at every size**, and a **single dequeue that costs 200,001** — two
per tipped element, plus the one pop that actually returned a value. The dequeue immediately after
it costs **1**.

All three numbers are true and together they are the point of the section. The amortised bound is a
statement about the *total* cost of a sequence; it says nothing about the worst individual call,
and the measurement puts five orders of magnitude between the average and the worst. NB-01 §2.2 made the same point about the
dynamic array's doubling resize — a single `append` can be $\Theta(n)$ — and it is worth seeing
twice, because "amortised $O(1)$" gets quoted as though it were "$O(1)$" constantly.

Where the distinction actually decides something:

- **Batch processing:** amortised is exactly the right measure. You care about the total.
- **A 16 ms frame budget or a hard real-time deadline:** it is the wrong measure. One
  $\Theta(n)$ operation blows the frame no matter how good the average is. This is why real-time
  allocators avoid amortised structures and why some game engines pre-size every container.
- **Tail latency in a service:** the p99 is measuring the worst case, not the average, so an
  amortised structure can look excellent on paper and produce exactly the latency spikes your
  dashboard is complaining about.

Why use this over a circular buffer? Mostly you would not — §1.3's ring is better in every
practical respect. The two-stack queue earns its place because **each stack can be an immutable
list**, giving a persistent queue with amortised $O(1)$ operations in a language with no mutation
at all. Okasaki's *Purely Functional Data Structures* builds it properly, including the real-time
variant that removes the worst case by tipping incrementally.

## 1.5 Java: `Stack`, `ArrayDeque`, `Queue` and `Deque`

Java's collections library contains a stack class you should not use, and the reason is a good
lesson in what §1.1 meant by *the restriction is the feature*.

`java.util.Stack` extends `Vector`. That is not an implementation detail — it is inheritance, so
**a `Stack` genuinely is a `List`**, and every `List` operation is part of its public interface.
The consequences are not subtle, and the cell below demonstrates them rather than describing them.

The folklore reason to avoid `Stack` is that `Vector`'s methods are `synchronized` and therefore
slow. That is worth testing rather than repeating, so the cell measures it too.

In [9]:
# ---------------------------------------------------------------------------
# What Stack lets you do, and whether it actually costs anything.
# ---------------------------------------------------------------------------
STACK_SRC = r"""
import java.util.*;

public class StackDemo {
    static double best(Runnable r, int reps) {
        double b = Double.MAX_VALUE;
        for (int i = 0; i < reps; i++) {
            long t0 = System.nanoTime();
            r.run();
            b = Math.min(b, (System.nanoTime() - t0) / 1e6);
        }
        return b;
    }

    public static void main(String[] args) {
        System.out.println("A. Stack extends Vector, so a Stack IS a List:");
        Stack<Integer> s = new Stack<>();
        s.push(1); s.push(2); s.push(3);
        System.out.println("   push 1,2,3   -> s.get(0)  = " + s.get(0)
                           + "        <- indexed access into a stack");
        System.out.println("                  s.toString = " + s
                           + "  <- bottom-to-top, the reverse of pop order");
        s.insertElementAt(99, 1);
        System.out.println("   s.insertElementAt(99, 1) -> " + s
                           + "   <- inserted into the MIDDLE of a stack");
        s.removeElementAt(0);
        System.out.println("   s.removeElementAt(0)     -> " + s
                           + "      <- removed from the BOTTOM");

        System.out.println();
        System.out.println("B. ArrayDeque offers none of that:");
        Deque<Integer> d = new ArrayDeque<>();
        d.push(1); d.push(2); d.push(3);
        System.out.println("   push 1,2,3   -> d.toString = " + d
                           + "  <- top first, matching pop order");
        System.out.println("                  no get(i), no insert-in-middle: they do not exist");

        System.out.println();
        System.out.println("C. Is Stack actually slower? n pushes then n pops, best of 11.");
        for (int w = 0; w < 400; w++) {                 // JIT warm-up
            Deque<Integer> wd = new ArrayDeque<>();
            Stack<Integer> ws = new Stack<>();
            for (int i = 0; i < 5000; i++) { wd.push(i); ws.push(i); }
            for (int i = 0; i < 5000; i++) { wd.pop(); ws.pop(); }
        }
        System.out.printf("   %10s %16s %14s %12s%n", "n", "ArrayDeque(ms)", "Stack(ms)", "Stack/AD");
        for (int trial = 0; trial < 3; trial++) {
            for (int n : new int[]{400_000, 800_000}) {
                final int N = n;
                double ad = best(() -> {
                    Deque<Integer> q = new ArrayDeque<>();
                    for (int i = 0; i < N; i++) q.push(i);
                    for (int i = 0; i < N; i++) q.pop();
                }, 11);
                double st = best(() -> {
                    Stack<Integer> q = new Stack<>();
                    for (int i = 0; i < N; i++) q.push(i);
                    for (int i = 0; i < N; i++) q.pop();
                }, 11);
                System.out.printf("   %10d %16.2f %14.2f %11.2fx%n", n, ad, st, st / ad);
            }
        }
    }
}
"""

if JAVA:
    print(run_java(STACK_SRC, timeout=900))
else:
    print("JDK not available; skipping the Java section.")

A. Stack extends Vector, so a Stack IS a List:
   push 1,2,3   -> s.get(0)  = 1        <- indexed access into a stack
                  s.toString = [1, 2, 3]  <- bottom-to-top, the reverse of pop order
   s.insertElementAt(99, 1) -> [1, 99, 2, 3]   <- inserted into the MIDDLE of a stack
   s.removeElementAt(0)     -> [99, 2, 3]      <- removed from the BOTTOM

B. ArrayDeque offers none of that:
   push 1,2,3   -> d.toString = [3, 2, 1]  <- top first, matching pop order
                  no get(i), no insert-in-middle: they do not exist

C. Is Stack actually slower? n pushes then n pops, best of 11.
            n   ArrayDeque(ms)      Stack(ms)     Stack/AD
       400000             4.69          13.30        2.83x
       800000             7.13           8.38        1.18x
       400000             3.08           3.90        1.27x
       800000             8.62          12.69        1.47x
       400000             3.40           3.09        0.91x
       800000             6.82         

**Part A is the argument.** `s.get(0)` reads the bottom of a stack. `insertElementAt(99, 1)` puts
an element in the *middle* of one. `removeElementAt(0)` takes one off the bottom. None of these
operations means anything for a stack, all of them compile, and each one lets a caller depend on
behaviour that any other stack implementation would be free to change. §1.1's principle —
**the caller cannot depend on what you did not promise** — inverted: `Stack` promised everything,
so it can never change anything.

Even `toString` is a trap: a `Stack` prints bottom-to-top, so `[1, 2, 3]` will pop `3` first. The
printed order is the reverse of the order you will get. `ArrayDeque` prints top-first.

**Part C corrects the folklore, and this is the interesting part.** The usual explanation for
avoiding `Stack` is that `Vector`'s synchronization makes it slow. Measured across three trials at
two sizes, **the ratio bounces either side of 1.0** — sometimes `Stack` is slower, sometimes
faster, mostly within noise. A modern JIT optimises uncontended locks aggressively, so in
single-threaded code the `synchronized` keyword costs close to nothing here, and allocation and GC
dominate the measurement entirely.

So the widely repeated performance argument does not survive contact with a stopwatch on a current
JVM. **The real case against `Stack` is the API**, and it is strong enough on its own. Being right
for the wrong reason is still worth correcting, because the wrong reason stops being true and the
right one does not.

**The interface guidance**, which is the practical takeaway:

| You want | Use | Notes |
|---|---|---|
| a stack | `ArrayDeque` via `Deque` | `push`/`pop`/`peek`; the JDK recommends it over `Stack` |
| a queue | `ArrayDeque` via `Queue` | `offer`/`poll`/`peek` |
| a double-ended queue | `ArrayDeque` via `Deque` | one class does all three |
| a thread-safe queue | `ArrayBlockingQueue`, `ConcurrentLinkedQueue` | not `Stack`, whose per-method locking gives no useful atomicity anyway |

**Declare the interface, not the class** — `Deque<Integer> d = new ArrayDeque<>();` — so the
restriction of §1.1 is enforced by the type. And note the sharp edge: `ArrayDeque` **rejects
`null`**, because it uses `null` internally to mark empty slots exactly as §1.3's `RingDeque` does.
`LinkedList` accepts `null`, so code that relied on it fails on the switch. That is the one
genuine migration hazard, and it comes straight from the circular buffer's implementation.

***
# Part 2 - Worked problems: the monotonic stack

Part 1 built the containers. This part is about the one technique that makes them indispensable,
and it is worth stating up front what makes it hard: **the monotonic stack has an invariant, and
almost nobody writes it down.** People learn the code shape — a `while` loop that pops before
pushing — and then cannot adapt it, because the shape is not the idea.

So this part derives the technique from the invariant, applies it four times, and Part 3 asserts
the invariant inside the loop.

| # | Problem | What it asks | Stack holds |
|---|---|---|---|
| 2.1 | Next greater element | the *index* of the next larger value | indices, values non-increasing |
| 2.2 | Daily temperatures | the same thing, phrased as a distance | indices, values non-increasing |
| 2.3 | Largest rectangle in a histogram | the widest span each bar can dominate | indices, values increasing |
| 2.4 | Sliding window maximum | the max of every window of size $k$ | indices, values decreasing (**deque**) |

## 2.1 The monotonic stack, derived

**The problem.** For each element, find the index of the next element to its right that is
strictly greater. `[2, 1, 3]` → `[2, 2, -1]`: the next greater than `2` is at index 2, likewise for
`1`, and `3` has none.

**The brute force** scans right from each position: $\Theta(n^2)$ worst case.

**The derivation.** Ask what the brute force wastes. Walking left to right, suppose we have seen
`5` and then `1`. If some later element is greater than `5`, it is certainly greater than `1` — so
`1` will be answered *no later* than `5` is. More importantly, once we have seen `1`, the `5`
behind it is still waiting and still relevant, but any element between them that was **smaller than
1** can never be the answer to anything we have not already answered.

That suggests keeping only the elements still waiting for an answer, and noticing that they are
necessarily in **decreasing** order — if an earlier waiting element were smaller than a later one,
the later one would have answered it.

> **The invariant.** The stack holds the indices of elements **not yet answered**, and their values
> are **non-increasing** from bottom to top.

Non-increasing rather than strictly decreasing, because we pop only on *strictly* less: two equal
values never evict each other, so they sit on the stack together. That distinction looks pedantic
and is not — §3.1 states the invariant as an assertion and this is exactly the clause it gets
wrong if you are careless.

Now the algorithm writes itself. For each new element `x`:

- while the top of the stack is smaller than `x`, `x` is its answer — pop it and record;
- push `x`'s index.

Anything left on the stack at the end has no next greater element.

The two-line reason this is $\Theta(n)$ rather than $\Theta(n^2)$, despite the nested loop:
**every index is pushed exactly once and popped at most once**, so the inner `while` executes at
most $n$ times *in total across the whole run* — not per iteration. That is an amortised argument
of exactly the shape as §1.4's, and Part 3 measures the constant.

In [10]:
# ---------------------------------------------------------------------------
# 2.1 Next greater element.
# ---------------------------------------------------------------------------
def next_greater(a):
    """For each i, the index of the next j > i with a[j] > a[i], else -1."""
    out = [-1] * len(a)
    stack = []                       # indices; a[stack] strictly decreasing
    for i, x in enumerate(a):
        while stack and a[stack[-1]] < x:
            out[stack.pop()] = i     # x is the answer for the popped index
        stack.append(i)
    return out                       # anything still on the stack keeps -1


def next_greater_brute(a):
    """Reference: scan right from every position."""
    out = []
    for i, x in enumerate(a):
        nxt = -1
        for j in range(i + 1, len(a)):
            if a[j] > x:
                nxt = j
                break
        out.append(nxt)
    return out


def gen_ints(rng):
    return [rng.randrange(-6, 7) for _ in range(rng.randrange(0, 22))]


checked = stress(next_greater, next_greater_brute, gen_ints, n=4000, seed=RANDOM_SEED,
                 label="next_greater")
print("next_greater: %s random arrays agree with the brute-force scan," % "{:,}".format(checked))
print("              including empty, single, all-equal and all-decreasing inputs.")
print()
for demo in ([2, 1, 3], [1, 2, 3], [3, 2, 1], [2, 2, 2]):
    print("  %-12s -> %s" % (demo, next_greater(demo)))
print()
print("  note [2,2,2] gives all -1: the comparison is STRICTLY greater, so equal")
print("  values do not answer each other. Whether ties count is a decision you")
print("  must make explicitly -- it is the < vs <= in the while condition.")

next_greater: 4,000 random arrays agree with the brute-force scan,
              including empty, single, all-equal and all-decreasing inputs.

  [2, 1, 3]    -> [2, 2, -1]
  [1, 2, 3]    -> [1, 2, -1]
  [3, 2, 1]    -> [-1, -1, -1]
  [2, 2, 2]    -> [-1, -1, -1]

  note [2,2,2] gives all -1: the comparison is STRICTLY greater, so equal
  values do not answer each other. Whether ties count is a decision you
  must make explicitly -- it is the < vs <= in the while condition.


The `<` versus `<=` in the while condition is the single most common source of wrong answers in
this family, and it is not a detail you can fix by staring at the code — it follows from whether
the problem says "greater" or "greater or equal". Write down which, then write the comparison.

Note also what the stack holds: **indices, not values.** You almost always want indices, because
the answer is usually a position or a distance (§2.2) or a width (§2.3), and you can always recover
the value with `a[i]`. Storing values throws away information you cannot get back.

## 2.2 Daily temperatures — the same algorithm wearing a different hat

**The problem.** Given daily temperatures, for each day say how many days until a warmer one, or 0
if there is none.

This is §2.1 with the answer expressed as a **distance** rather than an index. That is the entire
difference, and it is worth doing explicitly because recognising it is the skill: a large fraction
of "monotonic stack" problems are next-greater or next-smaller with the output reshaped.

In [11]:
# ---------------------------------------------------------------------------
# 2.2 Daily temperatures: next_greater, minus the index arithmetic.
# ---------------------------------------------------------------------------
def daily_temperatures(a):
    """Days until a strictly warmer temperature, 0 if none."""
    ng = next_greater(a)
    return [0 if j == -1 else j - i for i, j in enumerate(ng)]


def daily_temperatures_brute(a):
    out = []
    for i, x in enumerate(a):
        d = 0
        for j in range(i + 1, len(a)):
            if a[j] > x:
                d = j - i
                break
        out.append(d)
    return out


checked = stress(daily_temperatures, daily_temperatures_brute, gen_ints,
                 n=4000, seed=RANDOM_SEED, label="daily_temperatures")
print("daily_temperatures: %s random arrays agree with brute force." % "{:,}".format(checked))
print()
temps = [73, 74, 75, 71, 69, 72, 76, 73]
print("  temperatures:", temps)
print("  days to warmer:", daily_temperatures(temps))
print()
print("  Same stack, same loop, same invariant as 2.1 -- only the output changed.")

daily_temperatures: 4,000 random arrays agree with brute force.

  temperatures: [73, 74, 75, 71, 69, 72, 76, 73]
  days to warmer: [1, 1, 4, 2, 1, 1, 0, 0]

  Same stack, same loop, same invariant as 2.1 -- only the output changed.


## 2.3 Largest rectangle in a histogram

**The problem.** Bars of given heights, each one unit wide. Find the largest axis-aligned rectangle
that fits inside the histogram.

This is the hard one, and it is hard because the reframing is not obvious. The move:

> For each bar, the largest rectangle **whose height is that bar** extends left until a shorter bar
> and right until a shorter bar. Its area is `height * width`. The answer is the maximum over all
> bars.

That is a correct reduction — every maximal rectangle is limited in height by its shortest bar, so
considering each bar as "the shortest one" covers every candidate. And it turns the problem into:
for each bar, find the **nearest shorter bar on each side** — which is §2.1's problem with the
comparison flipped.

**The invariant here is increasing**, because we are looking for the nearest *smaller* element:

> The stack holds indices of bars **whose right boundary is not yet known**, with heights
> **increasing** from bottom to top.

When a bar arrives that is shorter than the top of the stack, it is the right boundary for that
top; the new top after popping is its left boundary. Both boundaries appear at once, which is what
makes it one pass.

**The trick that removes the end case:** iterate one step past the end with a virtual height of 0.
Since every real height is $\ge 0$, that flushes the entire stack, so bars still waiting at the end
need no special handling. This is §1.3's sentinel idea again — NB-04 §1.3 counted what sentinels
save, and here one saves an entire epilogue loop.

In [12]:
# ---------------------------------------------------------------------------
# 2.3 Largest rectangle in a histogram.
# ---------------------------------------------------------------------------
def largest_rectangle(h):
    """Maximum area of a rectangle fitting under the histogram."""
    stack = []              # indices; heights increasing bottom to top
    best = 0
    for i in range(len(h) + 1):
        cur = 0 if i == len(h) else h[i]        # virtual 0 bar flushes the stack
        while stack and h[stack[-1]] >= cur:
            top = stack.pop()
            left = stack[-1] if stack else -1   # nearest shorter bar on the left
            width = i - left - 1                # right boundary i, left boundary `left`
            best = max(best, h[top] * width)
        stack.append(i)
    return best


def largest_rectangle_brute(h):
    """Reference: every (i, j) span, tracking the running minimum height."""
    best = 0
    for i in range(len(h)):
        lo = h[i]
        for j in range(i, len(h)):
            lo = min(lo, h[j])
            best = max(best, lo * (j - i + 1))
    return best


def gen_heights(rng):
    return [rng.randrange(0, 9) for _ in range(rng.randrange(0, 16))]


checked = stress(largest_rectangle, largest_rectangle_brute, gen_heights,
                 n=4000, seed=RANDOM_SEED, label="largest_rectangle")
print("largest_rectangle: %s random histograms agree with the O(n^2) reference,"
      % "{:,}".format(checked))
print("                   including empty, all-zero, all-equal and single-bar inputs.")
print()
hist = [2, 1, 5, 6, 2, 3]
print("  heights:", hist, "-> largest area:", largest_rectangle(hist))
print("    (the 5 and 6 bars give 5 * 2 = 10)")
print()
print("  %-22s %s" % ("all equal [3,3,3,3]:", largest_rectangle([3, 3, 3, 3])))
print("  %-22s %s" % ("increasing [1,2,3,4]:", largest_rectangle([1, 2, 3, 4])))
print("  %-22s %s" % ("decreasing [4,3,2,1]:", largest_rectangle([4, 3, 2, 1])))

largest_rectangle: 4,000 random histograms agree with the O(n^2) reference,
                   including empty, all-zero, all-equal and single-bar inputs.

  heights: [2, 1, 5, 6, 2, 3] -> largest area: 10
    (the 5 and 6 bars give 5 * 2 = 10)

  all equal [3,3,3,3]:   12
  increasing [1,2,3,4]:  6
  decreasing [4,3,2,1]:  6


Two details that are easy to get wrong and hard to debug:

- **`width = i - left - 1`, not `i - top`.** The popped bar can extend *left* past its own index,
  all the way to just after the nearest shorter bar. Using `i - top` computes only the part to the
  right and silently under-reports.
- **`>=` rather than `>` in the while condition.** With equal heights, popping on `>=` means the
  left-hand duplicate gets a width that stops at its twin — an undercount for that bar. It is still
  correct overall, because the *rightmost* bar of a run of equals computes the full width and takes
  the maximum. Worth knowing that the intermediate values are wrong while the answer is right; it
  is why a test asserting per-bar widths would fail while a test asserting the area passes.

This reduction — "for each element, find the nearest smaller on both sides" — is the reusable part.
It also solves *maximal rectangle in a binary matrix* (run this per row on a histogram of column
heights), *sum of subarray minimums*, and *stock span*.

## 2.4 Sliding window maximum — the monotonic **deque**

**The problem.** For every window of $k$ consecutive elements, report the maximum.

The naive answer is $\Theta(nk)$. A heap gives $\Theta(n \log k)$ (NB-09). The right answer is
$\Theta(n)$, and it needs the structure Part 1 spent its time on: this is where a stack is not
enough and a **deque** is, because elements leave from **both** ends for different reasons.

> **The invariant.** The deque holds indices of elements that are still **candidates** to be the
> maximum of some current or future window, with values **decreasing** from front to back.

Two distinct removals, and keeping them straight is the whole problem:

- **From the back:** a new element that is $\ge$ the back makes the back permanently useless — it is
  older *and* smaller, so any window containing it also contains the newcomer. Pop it.
- **From the front:** the front may have fallen out of the window by age. Evict it by index.

The front of the deque is therefore always the maximum of the current window. And the amortised
argument is §2.1's: each index enters once and leaves once, so the total work is $\Theta(n)$
despite the inner loop.

In [13]:
# ---------------------------------------------------------------------------
# 2.4 Sliding window maximum, with a monotonic deque.
# ---------------------------------------------------------------------------
def window_max(payload):
    a, k = payload
    if k <= 0 or k > len(a):
        return []
    dq = deque()                     # indices; a[dq] decreasing front to back
    out = []
    for i, x in enumerate(a):
        while dq and a[dq[-1]] <= x:     # older AND smaller: useless forever
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:               # the front aged out of the window
            dq.popleft()
        if i >= k - 1:
            out.append(a[dq[0]])         # front is the window maximum
    return out


def window_max_brute(payload):
    a, k = payload
    if k <= 0 or k > len(a):
        return []
    return [max(a[i:i + k]) for i in range(len(a) - k + 1)]


def gen_window(rng):
    a = [rng.randrange(-6, 7) for _ in range(rng.randrange(0, 20))]
    return (a, rng.randrange(0, max(len(a), 1) + 2))


checked = stress(window_max, window_max_brute, gen_window, n=4000, seed=RANDOM_SEED,
                 label="window_max")
print("window_max: %s random (array, k) pairs agree with max() over every window,"
      % "{:,}".format(checked))
print("            including k = 0, k = 1, k = len(a) and k > len(a).")
print()
a, k = [1, 3, -1, -3, 5, 3, 6, 7], 3
print("  array:", a, " k =", k)
print("  window maxima:", window_max((a, k)))
print()
print("  Both ends are used, for different reasons: the BACK is popped because")
print("  an element became useless, the FRONT because it got too old. A stack")
print("  could do the first but not the second -- which is why this needs a deque.")

window_max: 4,000 random (array, k) pairs agree with max() over every window,
            including k = 0, k = 1, k = len(a) and k > len(a).

  array: [1, 3, -1, -3, 5, 3, 6, 7]  k = 3
  window maxima: [3, 3, 5, 5, 6, 7]

  Both ends are used, for different reasons: the BACK is popped because
  an element became useless, the FRONT because it got too old. A stack
  could do the first but not the second -- which is why this needs a deque.


The one-sentence test for whether a problem needs a deque rather than a stack: **does anything
leave the structure for a reason other than being superseded?** Here, ageing out of the window is
that other reason, and it happens at the opposite end.

Note that `<=` in the back-popping condition discards equal values. That is safe for a *maximum*
query — if two elements tie, keeping the newer one is always at least as good, because it survives
longer. If the problem asked you to count occurrences of the maximum, it would not be safe, and
that is the kind of change that quietly breaks a memorised template.

***
# Part 3 - The signature difficulty: the invariant nobody writes down

Every problem in Part 2 used the same six lines in a different arrangement, and that is exactly the
trap. People learn the **shape** — a `while` that pops before a push — and cannot adapt it, because
the shape is a consequence and not the idea.

The idea is an invariant. Part 2 stated one for each problem; this part does three things with it:
**assert it inside the loop** (§3.1), **prove the running time from it** (§3.2), and show that the
technique's real value is not speed but a **bounded cost that does not depend on the data**
(§3.3).

## 3.1 Asserting the invariant inside the loop

Everywhere else in this series, invariants are checked after each operation on a data structure.
A monotonic stack has no class to hang that on — it is three lines inside somebody's function — so
the invariant lives only in the author's head, and that is precisely why these problems are
error-prone.

So write it down and check it. The claim for §2.1 was:

> The stack holds indices of not-yet-answered elements, with values **non-increasing** from bottom
> to top.

There are two halves, and both are checkable at every step: the ordering, and the "not yet
answered" part — every index on the stack must still hold `-1` in the output.

**A note on how this section was written**, because it is the point of the section. The first draft
asserted *strictly* decreasing, and the assertion failed on `[0, 0]` within seconds: popping on
`<` never evicts an equal value, so duplicates coexist on the stack. The invariant caught the
notebook's own prose. That is the entire argument for writing invariants as code rather than as
comments — a comment that says "strictly decreasing" is wrong forever and silently.

In [14]:
# ---------------------------------------------------------------------------
# 3.1 The same algorithm, with its invariant asserted every iteration.
# ---------------------------------------------------------------------------
def stack_is_non_increasing(a, stack):
    vals = [a[i] for i in stack]
    return vals == sorted(vals, reverse=True)


def next_greater_checked(a):
    """Identical to next_greater, plus the invariant, checked at every step."""
    out = [-1] * len(a)
    stack = []

    def invariant(where):
        if not stack_is_non_increasing(a, stack):
            raise InvariantError("stack values %r are not non-increasing (%s)"
                                 % ([a[i] for i in stack], where))
        if stack != sorted(stack):
            raise InvariantError("stack indices %r are not increasing (%s)" % (stack, where))
        for i in stack:
            if out[i] != -1:
                raise InvariantError("index %d is on the stack but already answered (%s)"
                                     % (i, where))
        return True

    for i, x in enumerate(a):
        while stack and a[stack[-1]] < x:
            out[stack.pop()] = i
            invariant("after popping for i=%d" % i)
        stack.append(i)
        invariant("after pushing i=%d" % i)
    return out


checked = stress(next_greater_checked, next_greater_brute, gen_ints,
                 n=4000, seed=RANDOM_SEED, label="next_greater_checked")
print("next_greater_checked: %s arrays, with the invariant asserted after every"
      % "{:,}".format(checked))
print("                      push and every pop -- never violated.")

next_greater_checked: 4,000 arrays, with the invariant asserted after every
                      push and every pop -- never violated.


Now two bugs, chosen because they fail in completely different ways. Both are common; only one of
them is a *structural* error.

In [15]:
# ---------------------------------------------------------------------------
# Bug 1: `if` instead of `while` -- pops at most one element per step.
# ---------------------------------------------------------------------------
def next_greater_if(a):
    out = [-1] * len(a)
    stack = []
    for i, x in enumerate(a):
        if stack and a[stack[-1]] < x:        # <-- should be `while`
            out[stack.pop()] = i
        stack.append(i)
        if not stack_is_non_increasing(a, stack):
            raise InvariantError("stack values %r are not non-increasing "
                                 "(after pushing i=%d)" % ([a[j] for j in stack], i))
    return out


print("Bug 1 -- `if` instead of `while`:")
try:
    stress(next_greater_if, next_greater_brute, gen_ints, n=4000, seed=RANDOM_SEED,
           label="next_greater_if")
    print("  not caught")
except (StressFailure, InvariantError) as exc:
    for line in str(exc).split("\n"):
        print("  " + line)

print()
print("  smallest input that breaks it: [3, 2, 4]")
try:
    next_greater_if([3, 2, 4])
except InvariantError as exc:
    print("   ", exc)
print("    3 and 2 are on the stack; 4 pops only the 2, leaving [3, 4] -- increasing.")

Bug 1 -- `if` instead of `while`:
  next_greater_if: implementation disagrees with reference
    failing input : [-1, -5, 0]
    implementation: 'impl raised InvariantError: stack values [5, 6] are not non-increasing (after pushing i=2)'
    reference     : None

  smallest input that breaks it: [3, 2, 4]
    stack values [3, 4] are not non-increasing (after pushing i=2)
    3 and 2 are on the stack; 4 pops only the 2, leaving [3, 4] -- increasing.


In [16]:
# ---------------------------------------------------------------------------
# Bug 2: `<=` instead of `<` -- pops equal elements too.
# ---------------------------------------------------------------------------
def next_greater_le(a):
    out = [-1] * len(a)
    stack = []
    for i, x in enumerate(a):
        while stack and a[stack[-1]] <= x:    # <-- should be `<`
            out[stack.pop()] = i
        stack.append(i)
        if not stack_is_non_increasing(a, stack):
            raise InvariantError("stack values %r are not non-increasing "
                                 "(after pushing i=%d)" % ([a[j] for j in stack], i))
    return out


print("Bug 2 -- `<=` instead of `<`, with the SAME invariant check in place:")
try:
    stress(next_greater_le, next_greater_brute, gen_ints, n=4000, seed=RANDOM_SEED,
           label="next_greater_le")
    print("  not caught")
except (StressFailure, InvariantError) as exc:
    for line in str(exc).split("\n"):
        print("  " + line)

print()
print("  The invariant never fired. On [2, 2] the buggy version answers:")
print("    next_greater_le([2, 2]) =", next_greater_le([2, 2]),
      "  correct answer:", next_greater_brute([2, 2]))
print("  Its stack is strictly decreasing, which is still non-increasing --")
print("  a perfectly well-formed monotonic stack answering a different question.")

Bug 2 -- `<=` instead of `<`, with the SAME invariant check in place:
  next_greater_le: implementation disagrees with reference
    failing input : [0, 0]
    implementation: [2, 2, -1, 4, 7, 6, 7, -1, 12, 10, 12, 12, -1]
    reference     : [2, 2, -1, 4, 7, 6, 7, -1, -1, 10, 12, 12, -1]

  The invariant never fired. On [2, 2] the buggy version answers:
    next_greater_le([2, 2]) = [1, -1]   correct answer: [-1, -1]
  Its stack is strictly decreasing, which is still non-increasing --
  a perfectly well-formed monotonic stack answering a different question.


**The two bugs are caught by two different tools, and neither tool catches both.**

Bug 1 (`if` for `while`) breaks the structure, so the **invariant** fires — on a three-element
input, at the moment of the push, with the offending stack printed. You do not need a reference
implementation or a failing test case to find it; the algorithm reports its own inconsistency.

Bug 2 (`<=` for `<`) is caught only by the **differential test**. The invariant never fires,
because popping on `<=` leaves a *strictly* decreasing stack, which is still non-increasing — a
perfectly well-formed monotonic stack. It is simply answering a different question, "next greater
**or equal**", and no amount of structural checking will notice that the question changed.

The lesson generalises well beyond this notebook:

- **An invariant checks that the structure is consistent, not that it is solving your problem.**
  You need both an invariant and a reference implementation, and they catch disjoint classes of bug.
  NB-04 §1.4's tombstone bug is the mirror image: it was caught *only* by the invariant, because
  the counts all agreed.
- **The `<` versus `<=` choice is a specification question, not an implementation one.** Decide
  whether ties count *before* writing the loop, and put the answer in a comment. Half the wrong
  answers in this problem family are a correct algorithm for the adjacent problem.

## 3.2 Why it is linear, counted

The nested loop looks quadratic. It is not, and the argument is the accounting method again:

> Each index is pushed **exactly once**. Each index is popped **at most once**. The inner `while`
> body runs only when something is popped. Therefore the total number of inner iterations across
> the entire run is at most $n$ — not $n$ per outer step.

So the algorithm performs at most $2n$ stack operations regardless of input, which is $\Theta(n)$.
Counting them makes the constant explicit instead of hidden inside a $\Theta$.

In [17]:
# ---------------------------------------------------------------------------
# 3.2 Counting the stack operations.
# ---------------------------------------------------------------------------
def next_greater_counted(a):
    """Returns (pushes, pops) as well as the answer."""
    out = [-1] * len(a)
    stack = []
    pushes = pops = 0
    for i, x in enumerate(a):
        while stack and a[stack[-1]] < x:
            out[stack.pop()] = i
            pops += 1
        stack.append(i)
        pushes += 1
    return out, pushes, pops


print("Random input:")
print("  %10s %10s %10s %12s %14s" % ("n", "pushes", "pops", "total", "total / n"))
print("  " + "-" * 60)
for n in (10_000, 100_000, 1_000_000):
    rng = random.Random(n)
    a = [rng.randrange(0, 1_000_000) for _ in range(n)]
    _, p, q = next_greater_counted(a)
    print("  %10s %10s %10s %12s %14.3f"
          % ("{:,}".format(n), "{:,}".format(p), "{:,}".format(q),
             "{:,}".format(p + q), (p + q) / n))

print()
print("The two extreme inputs, which are extreme in OPPOSITE directions:")
print()
print("  %-22s %10s %10s %12s %12s" % ("input", "pushes", "pops", "total", "total / n"))
print("  " + "-" * 70)
for label, mk in (("strictly increasing", lambda n: list(range(n))),
                  ("strictly decreasing", lambda n: list(range(n, 0, -1))),
                  ("all equal", lambda n: [7] * n)):
    n = 100_000
    _, p, q = next_greater_counted(mk(n))
    print("  %-22s %10s %10s %12s %12.2f"
          % (label, "{:,}".format(p), "{:,}".format(q), "{:,}".format(p + q), (p + q) / n))

print()
print("Never more than 2 operations per element, whatever the data does.")

Random input:
           n     pushes       pops        total      total / n
  ------------------------------------------------------------
      10,000     10,000      9,992       19,992          1.999
     100,000    100,000     99,985      199,985          2.000


   1,000,000  1,000,000    999,986    1,999,986          2.000

The two extreme inputs, which are extreme in OPPOSITE directions:

  input                      pushes       pops        total    total / n
  ----------------------------------------------------------------------
  strictly increasing       100,000     99,999      199,999         2.00
  strictly decreasing       100,000          0      100,000         1.00
  all equal                 100,000          0      100,000         1.00

Never more than 2 operations per element, whatever the data does.


**At most 2 stack operations per element, always.** An increasing array pops nearly every element
(one push, one pop each: 2.00 per element). A decreasing array pops **nothing** — every element
just sits on the stack (1.00 per element). Random input lands in between at roughly 2.00, since
almost everything eventually gets popped.

The `while` loop that looks like it might run $n$ times per iteration cannot, because the elements
it consumes were each pushed once and are gone for good. This is the same argument as §1.4's
two-stack queue and NB-01 §2.2's dynamic array: **bound the total, not the individual step.**

Notice the shape of the result: the *distribution* of work moves around wildly with the input — all
of it in the pops for increasing data, none for decreasing — while the *total* barely moves. That
is worth one more measurement, because it is the technique's real selling point and it is not the
one usually advertised.

## 3.3 The point is not speed. It is that the cost is bounded.

The usual framing is "the monotonic stack turns $\Theta(n^2)$ into $\Theta(n)$", which is true only
for some inputs — and the exception is instructive enough to change how you think about the
technique.

The brute force and the stack have **opposite worst cases**:

- **Decreasing input.** No element ever has a next greater one, so the brute force scans to the end
  from every position: $\Theta(n^2)$. The stack pops nothing at all: its cheapest case.
- **Increasing input.** Every element's answer is the very next one, so the brute force's inner
  loop runs *once* and it is $\Theta(n)$ — as fast as the stack. Meanwhile the stack does its
  maximum work.

Both, measured on the same inputs.

In [18]:
# ---------------------------------------------------------------------------
# 3.3 Both algorithms, on both extremes.
# ---------------------------------------------------------------------------
for label, mk in (("DECREASING -- brute force's worst case",
                   lambda n: list(range(n, 0, -1))),
                  ("INCREASING -- the stack's worst case",
                   lambda n: list(range(n)))):
    print("=" * 66)
    print(label)
    print()
    print("  brute force (n = 1k to 8k, bounded by its quadratic case):")
    growth_table(measure_growth(next_greater_brute, [1_000, 2_000, 4_000, 8_000],
                                setup=mk, repeats=3))
    print("  monotonic stack (n = 100k to 800k, where it is slow enough to time):")
    growth_table(measure_growth(next_greater, [100_000, 200_000, 400_000, 800_000],
                                setup=mk, repeats=3))
    _, p, q = next_greater_counted(mk(100_000))
    print("  stack operations at n = 100,000: %s (%.2f per element)"
          % ("{:,}".format(p + q), (p + q) / 100_000))
    print()

DECREASING -- brute force's worst case

  brute force (n = 1k to 8k, bounded by its quadratic case):


         n        seconds      ratio
------------------------------------
     1,000       0.023087          -
     2,000       0.085098       3.69
     4,000       0.360153       4.23
     8,000       1.438652       3.99

best fit: O(n^2) (relative error 0.030); next: O(n log n) (1.093)
  monotonic stack (n = 100k to 800k, where it is slow enough to time):


         n        seconds      ratio
------------------------------------
   100,000       0.013587          -
   200,000       0.028563       2.10
   400,000       0.056907       1.99
   800,000       0.114111       2.01

best fit: O(n) (relative error 0.021); next: O(n log n) (0.048)
  stack operations at n = 100,000: 100,000 (1.00 per element)

INCREASING -- the stack's worst case

  brute force (n = 1k to 8k, bounded by its quadratic case):
         n        seconds      ratio
------------------------------------
     1,000       0.000287          -
     2,000       0.000687       2.40
     4,000       0.001129       1.64
     8,000       0.002360       2.09

best fit: O(n) (relative error 0.076); next: O(n log n) (0.136)
  monotonic stack (n = 100k to 800k, where it is slow enough to time):


         n        seconds      ratio
------------------------------------
   100,000       0.018613          -
   200,000       0.037028       1.99
   400,000       0.075759       2.05
   800,000       0.154024       2.03

best fit: O(n) (relative error 0.015); next: O(n log n) (0.048)
  stack operations at n = 100,000: 199,999 (2.00 per element)



On decreasing input the brute force is a clean $\Theta(n^2)$ and the stack is $\Theta(n)$ doing
**1.00 operations per element**. On increasing input the brute force is a clean $\Theta(n)$ — the
same class as the stack — while the stack does its maximum **2.00 per element**.

So the honest claim is not "the monotonic stack is faster". It is:

> **The monotonic stack's cost is bounded by $2n$ operations regardless of the input. The brute
> force's cost is a property of the data, ranging from $n$ to $n^2/2$.**

That is a better property than raw speed, and it is the same distinction the series keeps arriving
at from different directions. NB-02 §3 found naive string matching averaging 1.00 comparisons per
character on English text and 498 on constructed input. NB-03 §3 found a hash table at expected
$O(1)$ and adversarial $\Theta(n)$. Each time, the question is not "how fast is it?" but
**"who chooses the input, and what can they do to me?"**

If your data is genuinely random or genuinely friendly, the brute force may be fine and is
certainly easier to read. If the input arrives from outside, the bounded algorithm is the one that
lets you make a promise.

**How to recognise a monotonic-stack problem.** All four in Part 2 share a signature:

1. The answer for each element depends on the **nearest** element to one side satisfying a
   comparison — next/previous greater/smaller.
2. The obvious solution is a nested scan.
3. Once an element is "beaten", it can never be the answer for anything later.

That third condition is the one that licenses the technique, and it is what the invariant encodes:
**anything a newcomer beats can be discarded permanently.** If elements can become relevant again
later, a monotonic stack is the wrong tool.

The vocabulary, worth memorising as a table rather than four separate templates:

| You want | Comparison to pop on | Stack order | Scan |
|---|---|---|---|
| next greater | `stack top < x` | decreasing | left → right |
| next smaller | `stack top > x` | increasing | left → right |
| previous greater | `stack top <= x` | decreasing | left → right, read before push |
| previous smaller | `stack top >= x` | increasing | left → right, read before push |

And when elements also leave for a reason unrelated to being beaten — ageing out of a window —
the stack becomes a **deque** (§2.4).

***
# Part 4 - Tough questions

***

### Q1. Why do stacks and queues exist if an array can do both?

<details><summary>Answer</summary>

Because **the restriction is the product**. A stack is not a way of storing data — it is a dynamic
array with almost all of its interface removed, and removing the interface is what you are buying.

Three things follow from a narrow interface:

- **The caller cannot depend on what you did not promise.** A `push`/`pop`/`peek` interface can be
  an array today and a segmented buffer tomorrow. §1.5 shows what happens when a language gets this
  wrong: Java's `Stack` extends `Vector`, so `get(0)`, `insertElementAt(99, 1)` and
  `removeElementAt(0)` are all part of a stack's public API, and none of them means anything for a
  stack. Having promised everything, it can never change anything.
- **The restriction is the correctness argument.** "Are these brackets balanced?" is a one-sentence
  proof given a stack, because LIFO *is* the nesting discipline. You do not verify the algorithm so
  much as observe that the structure enforces it.
- **A narrow interface can be optimised behind.** Only ever touching the ends is exactly what
  licenses the circular buffer of §1.3 — you can move indices instead of data precisely because
  nobody can ask for element $i$.

The practical version: **declare the interface, not the class** — `Deque<Integer> d = new
ArrayDeque<>()` — so the discipline is enforced by the type system rather than by your memory.

</details>

***

### Q2. Implement a queue with an array. What goes wrong first?

<details><summary>Answer</summary>

The naive version is $\Theta(n)$ per dequeue and $\Theta(n^2)$ overall:

```python
q.append(x)     # Theta(1) amortised
q.pop(0)        # Theta(n) -- every remaining element shifts down one
```

§1.2 measured it: `list` fits $\Theta(n^2)$ cleanly, and against `deque` the ratio **doubles every
time $n$ doubles** — 13× at n=10,000 rising to 91× at n=80,000. `pop(0)` and `pop()` differ by one
character and by a complexity class, and nothing warns you. Java's `ArrayList.remove(0)` is the
same trap, and NB-04 §3.3 measured the same curve there.

**The fix is the circular buffer** (§1.3): keep a `head` index and a `size`, and let logical
position $i$ live at physical slot `(head + i) % capacity`. Dequeue advances `head` instead of
moving $n$ elements. Grow by doubling when full, unwrapping as you copy.

Three details that the naive implementation gets wrong:

1. **Clear the vacated slot** (`buf[head] = None`). Otherwise the buffer holds a reference to a
   popped element forever — a leak invisible to any test of *reachable* contents. §1.3's invariant
   has a clause specifically for this.
2. **Negative modulo.** `(head - 1) % capacity` works in Python; in Java and C, `-1 % 8` is `-1`
   and you get an out-of-bounds index. Use `(head - 1 + capacity) % capacity`.
3. **Distinguish full from empty.** Both give `head == tail` if you track two indices. Storing
   `size` instead of `tail` — as §1.3 does — makes the ambiguity disappear rather than handling it.

</details>

***

### Q3. Build a queue from two stacks. Prove the amortised bound.

<details><summary>Answer</summary>

`enqueue` pushes onto `inbox`. `dequeue` pops from `outbox`, and when `outbox` is empty, tips the
entire `inbox` into it first. Tipping reverses the order, which converts LIFO into FIFO.

**The proof (accounting method).** Follow one element through its whole life:

1. pushed onto `inbox` — once;
2. popped off `inbox` — at most once, during a tip;
3. pushed onto `outbox` — at most once, in the same tip;
4. popped off `outbox` — at most once.

**Four movements maximum, ever**, because an element is never tipped twice: once in the outbox it
stays until it leaves. So $n$ enqueues and $n$ dequeues cost $O(n)$ total — $O(1)$ amortised.

§1.4 counted it: **exactly 2.00 movements per operation** at every size from 1,000 to 1,000,000.

**And the part that matters more.** After 100,000 enqueues, the next dequeue costs **200,001
movements** and the one after it costs **1**. Amortised $O(1)$ is a promise about a *sequence*, not
about any individual call. That distinction decides real things: for batch work the average is the
right measure; for a 16 ms frame budget, a hard real-time deadline, or a p99 latency target it is
the wrong one, and a single $\Theta(n)$ operation ruins you no matter how good the mean is.

**Why bother, when §1.3's ring is better?** Because each stack can be an *immutable* list, which
gives a persistent queue with amortised $O(1)$ operations in a language with no mutation. Okasaki
builds it properly, including a real-time variant that tips incrementally to remove the worst case.

</details>

***

### Q4. What is a monotonic stack? State its invariant.

<details><summary>Answer</summary>

A stack whose contents are kept in sorted order by discarding anything a newcomer makes irrelevant.
For "next greater element" (§2.1):

> **The invariant.** The stack holds indices of elements **not yet answered**, with values
> **non-increasing** from bottom to top.

**Non-increasing, not strictly decreasing** — and this is not pedantry. Popping on `<` never evicts
an equal value, so duplicates coexist. §3.1's first draft asserted *strictly* decreasing and the
assertion failed on `[0, 0]` within seconds. The invariant caught the notebook's own prose, which
is the argument for writing invariants as executable code rather than as comments.

**The algorithm follows from the invariant.** For each `x`: while the top is smaller, `x` is its
answer — pop and record; then push. Anything left at the end has no answer.

**Why $\Theta(n)$ despite a nested loop:** each index is pushed exactly once and popped at most
once, so the inner `while` runs at most $n$ times *in total across the whole run*. §3.2 counted it —
never more than **2 operations per element**, whatever the data does.

**The licensing condition**, which tells you when the technique applies at all: once an element is
beaten, it can never be the answer for anything later. If elements can become relevant again, a
monotonic stack is the wrong tool.

| You want | Pop while | Stack order |
|---|---|---|
| next greater | `top < x` | non-increasing |
| next smaller | `top > x` | non-decreasing |
| previous greater / smaller | same, but read the top *before* pushing | same |

</details>

***

### Q5. The monotonic stack turns $\Theta(n^2)$ into $\Theta(n)$. Is that true?

<details><summary>Answer</summary>

**Only for some inputs, and the exception is the interesting part.** §3.3 measured both algorithms
on both extremes:

| Input | Brute force | Monotonic stack |
|---|---|---|
| **decreasing** | $\Theta(n^2)$ (nothing is ever answered, so every scan runs to the end) | $\Theta(n)$, **1.00 ops/element** — pops nothing |
| **increasing** | $\Theta(n)$ — every answer is the *very next* element | $\Theta(n)$, **2.00 ops/element** — its maximum |

They have **opposite worst cases**, and on increasing input the brute force is in the same
complexity class as the stack.

So the honest claim is not "the stack is faster". It is:

> **The stack's cost is bounded at $2n$ operations regardless of the input. The brute force's cost
> is a property of the data, ranging from $n$ to $n^2/2$.**

That is a better property than raw speed, and it is the same distinction this series keeps
reaching from different directions: NB-02 §3 measured naive string matching at 1.00 comparisons per
character on English and 498 on constructed input; NB-03 §3 measured a hash table at expected
$O(1)$ and adversarial $\Theta(n)$. Each time the real question is **"who chooses the input?"**

If the data is yours and friendly, the brute force may be fine and is easier to read. If it arrives
from outside, the bounded algorithm is the one that lets you make a promise.

</details>

***

### Q6. Sliding window maximum — why a deque and not a stack?

<details><summary>Answer</summary>

Because elements leave for **two unrelated reasons**, at opposite ends.

> **The invariant.** The deque holds indices still capable of being some window's maximum, values
> **decreasing** from front to back.

- **From the back:** a newcomer $\ge$ the back makes it permanently useless — the back is older
  *and* smaller, so every window containing it also contains the newcomer. Pop it.
- **From the front:** the front may simply have aged out of the window. Evict by index.

The front is therefore always the current window's maximum. A stack can express the first removal
but not the second, which is precisely why this problem needs the structure §1.3 built.

**Complexity:** $\Theta(n)$, by §3.2's argument — each index enters once and leaves once. A heap
gives $\Theta(n \log k)$ and the naive scan $\Theta(nk)$.

**The one-sentence test** for stack versus deque: *does anything leave for a reason other than
being superseded?* If yes, you need both ends.

**A trap in the template:** the back-popping condition uses `<=`, which discards ties. Safe for a
maximum — of two equal values the newer survives longer, so it is always at least as good. **Not**
safe if the problem asks how many times the maximum occurs. Memorised templates break on exactly
this kind of change, which is why §2.4 states the invariant rather than the code shape.

</details>

***

### Q7. Explain largest-rectangle-in-a-histogram.

<details><summary>Answer</summary>

**The reframing is the whole problem.** For each bar, the largest rectangle *of that bar's height*
extends left until a shorter bar and right until a shorter bar; its area is `height × width`. Every
maximal rectangle is limited by its shortest bar, so ranging over "each bar as the shortest one"
covers every candidate.

That converts the problem into: **for each bar, find the nearest shorter bar on both sides** —
§2.1's problem with the comparison flipped.

> **The invariant.** The stack holds indices of bars whose right boundary is unknown, with heights
> **increasing** from bottom to top.

When a shorter bar arrives it is the right boundary for the stack top; after popping, the new top
is the left boundary. Both appear at once, which is what makes it one pass. $\Theta(n)$ by §3.2's
argument.

**Two things that are easy to get wrong:**

- **`width = i - left - 1`, not `i - top`.** The popped bar extends left past its own index, all
  the way to just after the nearest shorter bar. Using `i - top` counts only the right half and
  silently under-reports.
- **Iterate one step past the end with a virtual height of 0**, which flushes the stack and removes
  the epilogue entirely. This is §1.3's sentinel idea, and NB-04 §1.3 counted what sentinels save.

**Where the reduction transfers:** maximal rectangle in a binary matrix (run this per row over
column heights), sum of subarray minimums, stock span.

</details>

***

### Q8. `Stack` vs `ArrayDeque` in Java — and is `Stack` actually slow?

<details><summary>Answer</summary>

**Use `ArrayDeque`. But the usual reason given for it is wrong**, and §1.5 measured that.

The folklore is that `Vector`'s `synchronized` methods make `Stack` slow. Measured across three
trials at two sizes with heavy JIT warm-up, the ratio **bounced either side of 1.0** — sometimes
slower, sometimes faster, mostly noise. A modern JIT optimises uncontended locks aggressively, so
single-threaded `synchronized` costs close to nothing here and allocation dominates the benchmark.

**The real case against `Stack` is its API**, and it is decisive on its own. `Stack extends Vector`,
so a `Stack` *is* a `List`:

```
s.push(1); s.push(2); s.push(3);
s.get(0)                  -> 1              indexed access into a stack
s.toString()              -> [1, 2, 3]      bottom-to-top: the REVERSE of pop order
s.insertElementAt(99, 1)  -> [1, 99, 2, 3]  inserted into the middle of a stack
s.removeElementAt(0)      -> [99, 2, 3]     removed from the bottom
```

Every one of those compiles, and each lets a caller depend on behaviour no other stack could
provide. Q1's principle inverted: having promised everything, it can never change anything. Even
`toString` misleads — `ArrayDeque` prints top-first, matching pop order.

**The guidance:** `Deque<T> d = new ArrayDeque<>()` for stacks, queues and deques alike; the
concurrent queues (`ArrayBlockingQueue`, `ConcurrentLinkedQueue`) when you need thread safety —
`Stack`'s per-method locking gives no useful atomicity anyway, since a `pop` still needs an
`isEmpty` check that is not part of the same lock.

**The one real migration hazard:** `ArrayDeque` rejects `null`, because it uses `null` internally
to mark empty slots — exactly as §1.3's `RingDeque` does. `LinkedList` accepts `null`, so code
relying on that breaks on the switch.

</details>

***

### Q9. Where do stacks appear that you did not put there?

<details><summary>Answer</summary>

Two of them run underneath every program you write.

**1. The call stack.** Each call pushes a frame holding parameters, locals and the return address;
each return pops. LIFO is exactly right because calls nest. Consequences you have already met:

- **Stack overflow is a full stack.** NB-04 §2.1 measured recursive list reversal dying at
  n = 1,000 in CPython. Depth $\Theta(\log n)$ is safe forever; depth $\Theta(n)$ is a crash
  waiting for a big input.
- **Any recursion can be made iterative with an explicit stack**, because you are simply managing
  by hand what the runtime was managing for you. That is the standard fix for deep recursion and
  the reason iterative tree traversals (NB-06) exist.

**2. The expression parser.** Operator precedence is a stack discipline — the shunting-yard
algorithm converts infix to postfix with one stack for operators and one output queue, and
evaluating postfix needs a stack alone.

Others worth knowing: **undo/redo** (two stacks), **backtracking** (NB-17 — the stack *is* the
search path), **DFS** (NB-20, an explicit stack replacing recursion), **browser history**, and
**matched-delimiter checking** in every compiler and editor.

Queues, correspondingly: **BFS** (NB-20), scheduler run queues, message brokers, buffering between
producer and consumer, and rate limiting.

The pattern worth noticing: **stack for depth, queue for breadth.** Swapping the container in a
graph traversal converts DFS into BFS with no other change, which is one of the cleanest
demonstrations that these structures are policies rather than storage.

</details>

***

### Q10. What are these structures bad at?

<details><summary>Answer</summary>

Everything they deliberately excluded — which is the point, but you should be able to name the
costs:

- **Random access.** No `get(i)`. A circular buffer *could* offer $\Theta(1)$ indexing (§1.3: the
  address is `(head + i) % cap`), and `ArrayDeque` deliberately does not expose it, because
  promising it would prevent a future implementation from doing something else. If you need
  indexing you want an array.
- **Searching.** $\Theta(n)$, and you have to destroy the structure to do it — popping everything
  and pushing it back. If you need membership, you want a hash set (NB-03) alongside.
- **Iterating without consuming.** Not part of the contract. Real implementations offer it as a
  convenience, and code that depends on it is depending on something the abstraction never promised.
- **Ordering by anything but arrival.** If you need "smallest first" rather than "first in" or
  "last in", you want a heap (NB-09) — a priority queue is a different structure with a
  confusingly similar name.
- **Amortised, not worst-case, guarantees.** §1.4's two-stack queue averages 2.00 movements and
  spikes to 200,001. §1.3's ring buffer amortises its doubling resize the same way. Under a hard
  latency bound, pre-size the container or use a structure with a worst-case guarantee.

The meta-answer: these structures trade away every access pattern except one, in exchange for
making that one cheap and the code that uses it obviously correct.

</details>

***

### Q11. Design a stack with $O(1)$ `min()`.

<details><summary>Answer</summary>

`push`, `pop`, `top` and **`min`** all in $O(1)$.

**The answer:** a second stack holding the minimum *at each level*. On push, push
`min(x, current_min)`; on pop, pop both. `min()` reads the top of the auxiliary stack.

The insight is that a stack's history is perfectly nested — when you pop back to a previous state,
the minimum is exactly what it was then. So the minimum can be *recorded* rather than recomputed,
which no amount of cleverness would let you do for a structure allowing arbitrary removal.

**The space optimisation** usually asked as a follow-up: push onto the auxiliary stack only when
`x <= current_min`, and pop it only when `x == current_min`. Use `<=` and not `<`, or duplicate
minima break — pop one and you lose the record for its twin. Same `<` versus `<=` decision as
§2.1, with the same consequence for ties.

**Why this does not extend to a queue.** A queue removes from the *other* end, so the nesting
argument fails — the element leaving is the oldest, and its departure can change the minimum
unpredictably. A min-queue needs either the two-stack construction of §1.4 (keeping a min-stack for
each half) or a **monotonic deque**, which is exactly §2.4's sliding window maximum with an
unbounded window. That connection is worth seeing: "queue with $O(1)$ min" and "sliding window
maximum" are the same problem.

</details>

***

### Q12. How do you recognise a monotonic-stack problem?

<details><summary>Answer</summary>

Three signals, all present in every problem in Part 2:

1. **The answer for each element depends on the nearest element to one side satisfying a
   comparison** — next/previous greater/smaller. Watch for "next warmer day", "nearest taller
   building", "how long until", "span".
2. **The obvious solution is a nested scan**, $\Theta(n^2)$.
3. **Once an element is beaten it can never matter again.** This is the licensing condition, and it
   is what the invariant encodes: anything a newcomer beats is discardable *permanently*. If
   elements can become relevant again later, the technique does not apply.

**Then the mechanical part.** Decide four things and the code writes itself:

- next or previous? (scan direction, or read-before-push)
- greater or smaller? (which way the stack is sorted)
- do ties count? (`<` versus `<=` — decide from the problem statement, not by trial and error)
- do you need the index, the value, or a width? (store indices; you can always recover the rest)

**The tell that it is really a deque problem:** something leaves for a reason other than being
beaten — usually a window. Then it is §2.4.

**And the honest caveat from §3.3:** confirm the brute force is actually too slow for your data
before reaching for this. On increasing input the naive scan is $\Theta(n)$ and considerably easier
to read. The stack's virtue is a bound that holds whatever the input does, which matters when you
do not control the input and matters much less when you do.

</details>

***

## Coding challenges

### Challenge 1 — a fixed-capacity ring buffer with overwrite

§1.3's buffer grows when full. Real ring buffers usually do not.

1. Modify `RingDeque` to a **fixed capacity** that overwrites the oldest element when full — the
   standard design for logging, audio and telemetry buffers.
2. Verify against a reference built on `collections.deque(maxlen=n)`, invariant checked after every
   operation.
3. Add a **single-producer single-consumer** variant using only two indices and no size field, so
   producer and consumer touch disjoint state. Explain why "distinguish full from empty" becomes
   hard again and how sacrificing one slot solves it.
4. Measure it against `deque(maxlen=n)` and explain the gap using NB-00 §1.3.

### Challenge 2 — the real-time two-stack queue

§1.4's queue is amortised $O(1)$ with a $\Theta(n)$ worst case. Remove the worst case.

1. Implement Okasaki's **real-time queue**: instead of tipping all at once, move one element per
   operation so the work is spread out.
2. Instrument it as §1.4 did and show the worst *single* operation is now $O(1)$ — the number that
   matters is the maximum, not the mean.
3. Measure total work against the simple version. You will pay for the guarantee; quantify it.
4. Say which you would ship for a batch job and which for a 16 ms frame budget, and why.

### Challenge 3 — the largest rectangle, three ways

§2.3 solved it with one stack. Do it two more ways and compare.

1. **Divide and conquer:** recurse on the minimum bar. Derive the recurrence and find its worst
   case (a sorted histogram makes it $\Theta(n^2)$).
2. **Prefix arrays:** compute nearest-smaller-left and nearest-smaller-right into two arrays with
   two separate passes, then combine. Same complexity, arguably clearer, twice the memory.
3. Verify all three against each other over thousands of random histograms, and measure them.
4. Then extend the stack version to **maximal rectangle in a binary matrix**, running it per row on
   a histogram of column heights — $\Theta(rows \times cols)$, and one of the best payoffs for
   understanding §2.3 properly.

***
# Part 5 - Practice

| # | Exercise | The technique | Difficulty |
|---|---|---|---|
| 1 | Balanced brackets | The stack's defining application | ★☆☆☆☆ |
| 2 | Evaluate reverse Polish notation | Stack as an evaluator | ★★☆☆☆ |
| 3 | Min stack | An auxiliary stack of history | ★★☆☆☆ |
| 4 | Decode a nested string | Two stacks for nesting | ★★★☆☆ |
| 5 | Asteroid collision | A stack with a three-way outcome | ★★★☆☆ |
| 6 | Trapping rain water | Monotonic stack, or two pointers | ★★★★☆ |
| 7 | Basic calculator with precedence | Shunting-yard | ★★★★☆ |
| 8 | Maximal rectangle in a binary matrix | §2.3 applied per row | ★★★★★ |

***

### 1. Balanced brackets

- **Brief:** push openers, and on a closer check it matches the top. Valid iff nothing mismatches
  and the stack ends empty.
- **Good result:** $\Theta(n)$, verified against a reference over thousands of random bracket
  strings including unbalanced and interleaved ones.
- **The trap:** the **empty stack on a closer** (`")("`), and a **non-empty stack at the end**
  (`"(("`). Both are rejections and beginners check only one. Then extend to strings containing
  other characters, and to quotes — which are *not* a stack problem, because a quote is its own
  opener and closer.

### 2. Evaluate reverse Polish notation

- **Brief:** push operands; on an operator, pop two, apply, push the result. RPN needs no
  precedence rules, which is the entire reason it exists.
- **Good result:** $\Theta(n)$, verified against `eval` on generated expressions.
- **The trap:** **operand order.** `pop()` gives the *right* operand first, so subtraction and
  division must be `b - a` where `a` was popped first. Also integer division semantics — Python's
  `//` floors toward negative infinity while most RPN specs truncate toward zero, so
  `-7 // 2 == -4` where the answer wanted is `-3`. NB-00 flagged this; here it changes the output.

### 3. Min stack

- **Brief:** Q11 — an auxiliary stack recording the minimum at each level.
- **Good result:** all four operations genuinely $O(1)$; differential-test against a reference that
  recomputes `min` over the whole stack.
- **The trap:** the space-optimised version, where you push only when `x <= current_min`. Use `<`
  and duplicate minima break. Then try the follow-up that admits no auxiliary stack (encode both
  values in one number) and note it fails on overflow — worth knowing as a party trick and not as
  code.

### 4. Decode a nested string

`"3[a2[c]]"` → `"accaccacc"`.

- **Brief:** two stacks — one for repeat counts, one for the string built so far. On `[`, push
  both and reset; on `]`, pop and splice.
- **Good result:** $\Theta(\text{output length})$, verified against a recursive reference.
- **The trap:** **multi-digit numbers** (`12[a]`), and remembering that the current string must be
  saved *before* descending. Note the recursive version is more readable and has the depth problem
  of NB-04 §2.1 — a genuine trade to state.

### 5. Asteroid collision

Positive values move right, negative left; on collision the larger survives, equal sizes destroy
both.

- **Brief:** a stack of survivors. A left-mover collides with the stack top only if the top is a
  right-mover.
- **Good result:** $\Theta(n)$, verified against a simulation.
- **The trap:** the **three-way outcome** — the new asteroid is destroyed, the top is destroyed
  (and you must keep comparing), or both are. Getting the loop-versus-break structure right here is
  the exercise. A useful variant of the monotonic pattern where "beaten" has three cases.

### 6. Trapping rain water

- **Brief:** three good solutions. **Monotonic stack** (decreasing) filling water layer by layer;
  **two pointers** in $\Theta(1)$ space; **prefix max arrays** in $\Theta(n)$ space and the easiest
  to explain.
- **Good result:** at least two, verified against each other over thousands of random height arrays.
- **The trap:** the stack version computes water in horizontal slabs, not columns, and the width is
  `i - stack[-1] - 1` — the same boundary arithmetic as §2.3, and wrong in the same way if you use
  `i - top`. Do §2.3 first.

### 7. Basic calculator with precedence

Evaluate `"3 + 5 * 2 - (4 / 2)"` without `eval`.

- **Brief:** **shunting-yard** — one stack for operators, one output queue for values, popping
  operators of higher or equal precedence before pushing. Then evaluate the postfix output with
  exercise 2.
- **Good result:** correct for `+ - * /`, parentheses and unary minus; verified against `eval` on
  generated expressions.
- **The trap:** **unary minus** (`-3 + 4` versus `5 - 3`) needs the tokeniser to know whether a `-`
  follows an operand or an operator. And right-associativity: `2^3^2` is 512, not 64, so
  associativity changes the "higher **or equal**" test.

### 8. Maximal rectangle in a binary matrix

Largest all-ones rectangle in a binary matrix.

- **Brief:** for each row, build a histogram of consecutive ones in each column ending at that row,
  and run §2.3 on it. $\Theta(rows \times cols)$.
- **Good result:** verified against a brute force over small random matrices, and the complexity
  measured.
- **The trap:** resetting a column's height to 0 on a zero rather than decrementing, and rebuilding
  the histogram from scratch per row (which costs a factor of `rows`). This is the best payoff in
  the notebook for having understood §2.3, and it is genuinely hard the first time.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapter 10.1 — "Stacks and queues".**
> Six pages, and it does the circular buffer properly — including the full-versus-empty ambiguity
> §1.3 sidesteps by storing `size`. The exercises on implementing a queue with two stacks and a
> stack with two queues are worth doing before reading §1.4.

**2. [`ArrayDeque` Javadoc](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/ArrayDeque.html)** —
**Free.**
> Short, and unusually direct for API documentation: *"likely to be faster than `Stack` when used
> as a stack, and faster than `LinkedList` when used as a queue."* Two recommendations against
> other classes in the same library. It also documents the `null` restriction §1.5 flags, and the
> reason for it is exactly §1.3's sentinel slot.

**3. [Chris Okasaki, *Purely Functional Data Structures*](https://www.cs.cmu.edu/~rwh/students/okasaki.pdf)** —
the 1996 thesis. **Free.**
> Chapters 5–6 are where §1.4's two-stack queue becomes a serious idea: the amortised bound, why it
> *breaks* under persistence (an old version can be re-run repeatedly, paying the worst case each
> time), and the real-time variant that fixes it by tipping incrementally. The best available
> treatment of amortised analysis meeting immutability.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1–1.2 — the structures, the naive queue | **CLRS ch. 10.1**; **Sedgewick & Wayne §1.3** | 🔍 |
| 1.3 — the circular buffer | **CLRS ch. 10.1**; CPython's `Modules/_collectionsmodule.c`; the JDK's `ArrayDeque` | ✅ (the source) |
| 1.4 — amortised analysis | **CLRS ch. 17**, the accounting method; **Okasaki ch. 5** | 🔍 / ✅ |
| 1.5 — why `Stack` is legacy | **`ArrayDeque` Javadoc**; **Bloch, *Effective Java*** on inheritance vs composition — `Stack extends Vector` is his cautionary example | ✅ (Javadoc) |
| 2.1–2.3 — monotonic stacks | No canonical paper; **Sedgewick** on the stock-span problem is the classic textbook framing | 🔍 |
| 2.4 — the monotonic deque | The sliding-window-minimum technique, standard in competitive programming | 🔍 |
| 2.3 — largest rectangle | **Chazelle**'s and later linear-time formulations; the stack version is folklore | 🔍 |
| 3 — bounded vs data-dependent cost | **Crosby & Wallach**, *Denial of Service via Algorithmic Complexity Attacks* — the same argument NB-03 §3 is built on | ✅ |
| Q9 — shunting-yard | **Dijkstra**, 1961, *Algol 60 translation* | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Okasaki, chapters 5–6.** §1.4 counts 2.00 movements per operation and calls it amortised
$O(1)$; Okasaki shows that claim is *false* the moment the queue is persistent, because an
adversary can hold a reference to the expensive state and re-run the tip repeatedly, paying
$\Theta(n)$ every time. The amortised argument silently assumed each state is used once.

That is the most useful thing in this notebook's reading list: an amortised bound is a statement
about a *sequence of operations on a single-threaded, single-use structure*, and three separate
things can invalidate it — persistence (Okasaki), a hard latency bound (§1.4), and an adversary
choosing the sequence (NB-03 §3). Read it and you will stop treating "amortised $O(1)$" as a
synonym for "$O(1)$", which §1.4 measured a 200,001-to-1 gap behind.

Then read **CLRS 17** for the accounting method done formally, and the **`ArrayDeque` Javadoc**
because it is two minutes and settles §1.5.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Queue is quadratic | `list.pop(0)` / `ArrayList.remove(0)` shifts everything (§1.2) | `collections.deque` / `ArrayDeque` |
| Ring buffer holds dead objects | Vacated slot not cleared (§1.3) | `buf[head] = None` on pop; assert the empty-slot clause |
| `IndexOutOfBounds` on `push_front` | `(head - 1) % cap` is negative in Java/C (§1.3) | `(head - 1 + cap) % cap` |
| Cannot tell a full buffer from an empty one | Two indices are ambiguous (§1.3) | Store `size`, not `tail`; or sacrifice one slot |
| Latency spikes despite "amortised $O(1)$" | One operation can still be $\Theta(n)$ (§1.4) | Pre-size, or use a worst-case structure |
| `NullPointerException` after switching to `ArrayDeque` | It rejects `null`; `LinkedList` did not (§1.5) | Use a sentinel object, or `Optional` |
| Monotonic stack gives wrong answers on ties | `<` versus `<=` (§2.1, §3.1) | Decide from the problem statement, comment it |
| Monotonic stack answers are partly right | `if` instead of `while` — only one pop per step (§3.1) | `while`; the invariant catches this one |
| Largest rectangle under-reports | `width = i - top` instead of `i - left - 1` (§2.3) | The popped bar extends left past its own index |
| Bars left on the stack at the end | No flush after the loop (§2.3) | Iterate one step past the end with a virtual 0 |
| Sliding window returns stale maxima | Front never evicted by age (§2.4) | `if dq[0] <= i - k: dq.popleft()` |
| Recursion overflows the stack | Depth $\Theta(n)$ (Q9, NB-04 §2.1) | Convert to an explicit stack |
| `Stack.toString()` looks backwards | It prints bottom-to-top (§1.5) | Use `ArrayDeque`, which prints top-first |

## Checklist

- [ ] Is any queue built on a plain list with `pop(0)` / `remove(0)` (§1.2)?
- [ ] Does the ring buffer **clear** vacated slots (§1.3)?
- [ ] Is negative modulo handled for the language you are in (§1.3)?
- [ ] Is "amortised $O(1)$" being relied on where a **per-operation** bound is needed (§1.4)?
- [ ] Is the interface declared (`Deque`), not the class (§1.5)?
- [ ] Can `null` reach an `ArrayDeque` (§1.5)?
- [ ] For a monotonic stack: is the **invariant written down**, and is `<` vs `<=` a decision or an
      accident (§2.1, §3.1)?
- [ ] Is it `while` and not `if` (§3.1)?
- [ ] Does anything leave the structure for a reason other than being beaten — if so, deque not
      stack (§2.4)?
- [ ] Was the total work **counted**, not assumed (§3.2)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `trees_zero_to_hero.ipynb` | Iterative traversal is an explicit stack; BFS is a queue |
| `heaps_zero_to_hero.ipynb` | The priority queue — "smallest first" rather than "first in", Q10's gap |
| [`linked_lists_zero_to_hero.ipynb`](linked_lists_zero_to_hero.ipynb) | §3.3's `ArrayDeque`-beats-`LinkedList` result, which §1.3 explains |
| [`arrays_zero_to_hero.ipynb`](arrays_zero_to_hero.ipynb) | The dynamic array the stack rides on, and its doubling argument |

See [`README.md`](README.md) for the full roster and reading order.